In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%%RecordEventWithColumnInfo
from pathlib import Path
# 
import numpy as np
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time

In [ ]:
%%RecordEventWithColumnInfo
passmark = 40

In [ ]:
%%RecordEventWithColumnInfo
### cell 0 ###

benchmark_name = "student-performance-in-exams"
df = pd.read_csv(Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "StudentsPerformance.csv")
factor = 10
df = pd.concat([df]*factor)
df.info()

In [ ]:
%%RecordEventWithColumnInfo
### cell 1 ###

df.isna().sum()

In [ ]:
%%RecordEventWithColumnInfo
### cell 2 ###

df.head()

In [ ]:
%%RecordEventWithColumnInfo
### cell 3 ###

df.describe()

In [ ]:
%%RecordEventWithColumnInfo
### cell 4 ###

df.isnull().sum()

In [ ]:
%%RecordEventWithColumnInfo
### cell 5 ###

df['math score'] = pd.to_numeric(df['math score'], errors='coerce')
df['Math_PassStatus'] = np.where(df['math score']<passmark, 'F', 'P')
df.Math_PassStatus.value_counts()

In [ ]:
%%RecordEventWithColumnInfo
### cell 6 ###

df['reading score'] = pd.to_numeric(df['reading score'], errors='coerce')
df['Reading_PassStatus'] = np.where(df['reading score']<passmark, 'F', 'P')
df.Reading_PassStatus.value_counts()

In [ ]:
%%RecordEventWithColumnInfo
### cell 7 ###

df['writing score'] = pd.to_numeric(df['writing score'], errors='coerce')
df['Writing_PassStatus'] = np.where(df['writing score']<passmark, 'F', 'P')
df.Writing_PassStatus.value_counts()

In [ ]:
%%RecordEventWithColumnInfo
### cell 8 ###

df['OverAll_PassStatus'] = df.apply(lambda x : 'F' if x['Math_PassStatus'] == 'F' or 
                                    x['Reading_PassStatus'] == 'F' or x['Writing_PassStatus'] == 'F' else 'P', axis =1)

df.OverAll_PassStatus.value_counts()

In [ ]:
%%RecordEventWithColumnInfo
### cell 9 ###

df['Total_Marks'] = df['math score']+df['reading score']+df['writing score']
df['Percentage'] = df['Total_Marks']/3

In [ ]:
%%RecordEventWithColumnInfo
### cell 10 ###

def GetGrade(Percentage, OverAll_PassStatus):
    if ( OverAll_PassStatus == 'F'):
        return 'F'    
    if ( Percentage >= 80 ):
        return 'A'
    if ( Percentage >= 70):
        return 'B'
    if ( Percentage >= 60):
        return 'C'
    if ( Percentage >= 50):
        return 'D'
    if ( Percentage >= 40):
        return 'E'
    else: 
        return 'F'

df['Grade'] = df.apply(lambda x : GetGrade(x['Percentage'], x['OverAll_PassStatus']), axis=1)

df.Grade.value_counts()